# RUIP_BA_Swipe: Valid claim and Invlid claim 
This program is primarily configured and tuned for the swipe dataset (applying RP+Laplace noise), and most model parameters and hyperparameters are optimized for this setting. When applying the same framework to other datasets, these parameters generally need to be adjusted and re-tuned to ensure optimal performance.

The program processes oversampled training data from Group 1 by first projecting it into a lower-dimensional space using random projection. To ensure differential privacy, either Laplace or Gaussian noise is added to the projected features. The resulting privacy-preserved data is then used to train a machine learning model.

For the valid claim setting, the trained model is evaluated using test data from Group 1. The test data is transformed using the same random projection matrices and the same differential privacy configuration as used during training, ensuring full consistency between training and testing procedures.

For the invalid claim setting, the trained model is evaluated using test data from Group 2. In this case, a different random projection matrix is used for Group 2, introducing inconsistency in the feature transformation relative to the training pipeline. However, the differential privacy parameters (e.g., $\varepsilon$, $\delta$, and noise scale) remain unchanged, as they are treated as fixed and publicly known across both groups.

Overall, this program supports both valid and invalid claim settings across different datasets. It can operate on plain data as well as in configurations involving random projection and/or differential privacy noise (Laplace or Gaussian), with appropriate adjustment of model parameters and hyperparameters.

In [1]:
# =====================================================
# Hyperparameters and Configuration for model training
# =====================================================

# Dataset paths
DATASET_PATH = 'Dataset/OversampledSwipeData.csv'
TESTDATASET_PATH = 'Dataset/SwipeDatatest.csv'

# Profile separation
SEPARATE_PROFILE = 68

# Random projection parameters
USE_RANDOM_PROJECTION = True

if USE_RANDOM_PROJECTION==False:
    N_COMPONENTS = 34
else:
   N_COMPONENTS = 30
   

# Differential privacy parameters
USE_LAPLACE_NOISE = True
USE_GAUSSIAN_NOISE = False
EPSILON = 7.0
SENSITIVITY = 1.0
DELTA = 1e-5

# Input dimension for neural network
INPUT_DIM = N_COMPONENTS
BATCH_SIZE = 32
EPOCHS = 50

# Dynamic column generation
column1 = [f'RPF{i+1}' for i in range(N_COMPONENTS)]

# =====================================================
# Load dataset
# =====================================================

import pandas as pd
import numpy as np

dataset = pd.read_csv(DATASET_PATH, index_col=0)

In [2]:
# Separate the profiles into two groups:
# (i) training profiles (0-SEPARATE_PROFILE-1)
# (ii) auxiliary profiles (SEPARATE_PROFILE and above)

totalUser = len(pd.unique(dataset['Label']))

trainingData = dataset[dataset['Label'] < SEPARATE_PROFILE]
auxilaryData = dataset[dataset['Label'] >= SEPARATE_PROFILE]

print("Total user in training dataset:", len(pd.unique(trainingData['Label'])))
print("Total user in auxiliary dataset:", len(pd.unique(auxilaryData['Label'])))

Total user in training dataset: 68
Total user in auxiliary dataset: 18


In [3]:
#Random Project
from sklearn.random_projection import SparseRandomProjection

def apply_random_projection(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):  
        rng = np.random.RandomState(seed)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [4]:
# Laplace noise
def add_laplace_noise(data, total_user, epsilon=EPSILON, sensitivity=SENSITIVITY):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        scale = sensitivity / epsilon
        noise = np.random.laplace(loc=0.0, scale=scale, size=Xdata.shape)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [5]:
# Gaussioan noise
def add_gaussian_noise(data, total_user,epsilon=EPSILON,sensitivity=SENSITIVITY,delta=DELTA):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        sigma = (sensitivity * np.sqrt(2 * np.log(1.25 / delta))) / epsilon
        noise = np.random.normal(0, sigma)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [6]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = trainingData
total_user=len(pd.unique(trainingData['Label']))
# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projection(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)


# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_16948\1979874663.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_16948\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [7]:
#Prepare the traning data for training and testing the model
import tensorflow
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

if USE_RANDOM_PROJECTION==False:
    X=trainingData.drop(columns=['Label'])
    y=trainingData['Label']
else:
    X=processed_dataset.drop(columns=['Label'])
    y=processed_dataset['Label']

Xtrain, Xval, ytrain, yval = train_test_split(X, y, test_size=0.2, random_state=22)

ytrain = to_categorical(ytrain)
yval = to_categorical(yval)
print(Xtrain.shape)
print(ytrain.shape)
print(Xval.shape)
print(yval.shape)

(16320, 30)
(16320, 68)
(4080, 30)
(4080, 68)


In [8]:
# import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#matplotlib inlineimport keras
from keras.layers import Dense, Dropout, Input,Activation,Dropout, Flatten
from keras.models import Model,Sequential
from keras.datasets import mnist

from keras.layers import BatchNormalization
from keras.optimizers import SGD, RMSprop, Adam
def adam_optimizer():
    return Adam(learning_rate=0.0002, beta_1=0.5)

def RMSprop_optimizer():
    return RMSprop(learning_rate=0.001, rho=0.9)

In [9]:
#neural network architecture

def create_classifierRP(release=False, totalClass=SEPARATE_PROFILE):
  classifier = Sequential()
  classifier.add(Dense(64, input_dim=N_COMPONENTS))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.5))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))


  classifier.add(Dense(64))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  classifier.add(Dense(totalClass, activation='softmax'))

  classifier.compile(loss='categorical_crossentropy', optimizer=RMSprop_optimizer(),metrics=['accuracy'])
  return classifier

Clasf=create_classifierRP()
Clasf.summary()

C:\Users\mdmor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 68)             │         4,420 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,004 (93.77 KB)

 Trainable params: 23,492 (91.77 KB)

 Non-trainable params: 512 (2.00 KB)

In [10]:
#Train the classifier
from keras.callbacks import ReduceLROnPlateau

learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=5, verbose=1, factor=0.5,min_lr=0.0001)
callbacks_list = [learning_rate_reduction]

Classfier2= create_classifierRP(True,68)

#------Comment will start from here
lossc='categorical_crossentropy'
optimizerc=RMSprop(learning_rate=0.001, rho=0.9)
Classfier2.compile(loss=lossc, optimizer=optimizerc,metrics=['accuracy'])
#------Comments will end from here
historyc2 =  Classfier2.fit(Xtrain, ytrain, batch_size=64, epochs=200, validation_data=(Xval, yval),verbose=1)

Epoch 1/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1398 - loss: 3.7424 - val_accuracy: 0.7721 - val_loss: 2.2527
Epoch 2/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4937 - loss: 2.1469 - val_accuracy: 0.9150 - val_loss: 0.8430
Epoch 3/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6245 - loss: 1.4466 - val_accuracy: 0.9368 - val_loss: 0.4247
Epoch 4/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6782 - loss: 1.1913 - val_accuracy: 0.9539 - val_loss: 0.2882
Epoch 5/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7083 - loss: 1.0312 - val_accuracy: 0.9586 - val_loss: 0.2207
Epoch 6/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7117 - loss: 0.9749 - val_accuracy: 0.9652 - val_loss: 0.1778
Epoch 7/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7352 - loss: 0.8934 - val_accuracy: 0.9708 - val_loss: 0.1550
Epoch 8/200
255/255 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7474 - loss: 0.8616 - val_accu

In [17]:
# =====================================================
# Hyperparameters and Configuration for valid and invalid claim
# =====================================================
#Type of claim
VALID_CLAIM = False
INVALID_CLAIM =True

In [18]:
#Random Project for vlaid and invalid claim
from sklearn.random_projection import SparseRandomProjection

def apply_random_projectionClaim(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user): 
        if VALID_CLAIM==True: 
            rng = np.random.RandomState(seed)
        else:
             rng = np.random.RandomState(seed+SEPARATE_PROFILE)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [19]:
#read the test data either for valid or invalid claim
import csv
import pandas as pd

if VALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] < SEPARATE_PROFILE]
    
if INVALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] >= SEPARATE_PROFILE]
    newID = np.random.randint(0, SEPARATE_PROFILE, size=testdataset.shape[0])
    testdataset['Label'] = newID
    
testdataset.head()

,1,2,3,4,5,6,7,8,9,10,...,25,26,27,28,29,30,31,32,33,Label
8796,0.013165,0.510668,0.732085,0.775555,0.716926,0.027529,0.024510,0.119330,0.654402,-0.149844,...,-0.048378,0.018561,0.000345,1.000000,0.860215,0.943739,0.00000,0.000000,0.97561,63
8209,0.036204,0.510668,0.681730,0.917961,0.661678,0.524333,0.333333,-0.000003,0.307162,-0.126702,...,-0.013927,0.026554,0.000705,0.180392,0.452035,0.276953,0.37244,0.138711,0.18747,52
8721,0.051381,0.223365,0.423734,0.417552,0.431169,0.000000,0.000000,0.013251,0.142010,-0.062371,...,0.002469,0.013271,0.000176,1.000000,0.860215,0.943739,0.00000,0.000000,0.97561,46
8875,0.016639,0.498441,0.925113,0.880265,0.908020,0.023247,0.031863,0.066555,0.697454,-0.141167,...,-0.034539,0.022292,0.000497,1.000000,0.860215,0.943739,0.00000,0.000000,0.97561,18
9294,0.029256,0.412619,0.587617,0.740061,0.507532,0.051660,0.039216,0.000593,0.386902,-0.345556,...,0.003056,0.070412,0.004958,1.000000,0.860215,0.943739,0.00000,0.000000,0.97561,27


In [20]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = testdataset
total_user=len(pd.unique(testdataset['Label']))


# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projectionClaim(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)

# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_16948\432162272.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_16948\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [21]:
if USE_RANDOM_PROJECTION==False:
    Xtest=testdataset.drop(columns=['Label'])
    ytest=testdataset['Label']
else:
    Xtest=processed_dataset.drop(columns=['Label'])
    ytest=processed_dataset['Label']
    
ytest = to_categorical(ytest)

In [22]:
#Performance of the classifier
Classfier2.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
loss, accuracy = Classfier2.evaluate(Xtest, ytest)
print('Loss:', loss)
print('Accuracy:', accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0111 - loss: 9.7269       
Loss: 9.575634002685547
Accuracy: 0.0117647061124444
